### 初始化

In [ ]:
# In[2]:

import os
import json
import copy
from JoinAgent import *

# 初始化API
llm = SimpleLLM()
parser = LLMParser()
divider=TextDivider(threshold=4096,overlap=0)  

num_threads = 500
checkpoint=100

### 01 对markdown进行修改

In [ ]:
# In[4]:


data_template01='''
=start_pad= ...修改后内容... =end_pad=
'''
prompt_template01 ='''
你是一个工作细致，数理能力很好的助手。我将给你一段数学类文本，请你帮我检查其中是否有编译错误，并帮我改正，并把改正后的内容按格式要求输出。
你的工作要求如下：
1、根据上下文语境，检索是否有上下文无关的错误符号出现，并加以修正。
2、修改过程中，应在确保修改完内容正确的基础上，做出尽量少的修改。
3、请你除了输出要求的内容，不输出其他任何内容。

具体输出格式：
{data_template}

以下是一个输出示例：
1+1=2

以下是我给你的文本：{pos1}，请你帮我进行检查，并输出存储修改后内容的文本。
'''

def validation01(text):
    return True

correction_prompt= '''
    你是一个严谨的校对员。我将给你一个由大模型生成的数据结构，请你根据规定格式内容进行校对和修正。

    校对的格式是：
    {data_template}

    以下是待校验的文本：{answer}，请你帮我校对和修正它。
    '''

empty_template01='''
[]
'''


In [ ]:
# In[5]:

# 使用parse_pads解析，调整数据结构
def pad_to_dict(a_dict):
    a_dict_1 = copy.deepcopy(a_dict)
    new_dict = {}
    for key, value in a_dict_1.items():
        split_values = value.split('.')
        formatted_values = {i: f'r"""{v.strip()}."""' for i, v in enumerate(split_values) if v.strip()}
        new_dict[key] = {'pos1':formatted_values}
    return new_dict


### 02 寻找题目隔断点

In [ ]:
# In[6]:


data_template02 ='''
[断点标号1,断点标号2,...]
'''

prompt_template02 ='''
你是一个工作细致的助手。我将给你一份数学教材，请你帮我识别出哪些属于同一个定义、定理、引理、推论、命题，每一个称为一个数学内容，并输出每条数学内容的最后一个短句对应的索引（即断点标号），并统一放入一个列表中。

注意：
1、对于最终分隔的结果，要考察是否分隔出了完整的数学内容，也就是数学内容中所有的数学对象是否都有清晰明白的定义。
2、短句的索引是按顺序的。尽量少输出标号，除非你有把握没有切断一个数学内容。
3、请你除了输出这个列表外，不要输出其他任何内容。

具体提取的格式是：
{data_template}

以上是示例，请你不要在正文中输出
############################################

以下是我给你的短句及其索引：{pos1}，请你帮我提取出所有的断点标号，并放入一个列表。
'''

def validation02(text):
    return True

correction_prompt= '''
    你是一个严谨的校对员。我将给你一个由大模型生成的数据结构，请你根据规定格式内容进行校对和修正。

    校对的格式是：
    {data_template}

    以下是待校验的文本：{answer}，请你帮我校对和修正这段内容。
    '''

empty_template02='''
[]
'''


### 组合每道题目的信息

In [ ]:
# In[7]:


def extract_problem(chopped_dict, marker_dict):
    chopped_dict_1 = copy.deepcopy(chopped_dict)
    for key, value in marker_dict.items():
        for cut_mark in value:
            if cut_mark in chopped_dict[key]['pos1']:
                original_string = chopped_dict[key]['pos1'][cut_mark].strip('r"""')
                chopped_dict_1[key]['pos1'][cut_mark] = original_string + "###cut mark###"
            else:
                print(f"Warning: cut_mark {cut_mark} not in chopped_dict[{key}]['pos1']")

    # 把所有字符串拼接起来 
    concatenated_string = ""
    for key in sorted(chopped_dict_1.keys()):
        pos1_dict = chopped_dict_1[key].get('pos1', {})
        for sub_key in sorted(pos1_dict.keys()):
            concatenated_string += pos1_dict[sub_key]
    
    # 根据断点分隔字符串
    split_list = concatenated_string.split("###cut mark###")
    split_list = [value for value in split_list if value]  # 删除空元素
    split_dict = {index: {'pos1': value} for index, value in enumerate(split_list)}
    
    return split_dict



### 03 提取内容

In [ ]:
# In[8]:


data_template03 ='''{
0:{
  "env": "env1",
  "title": "title1",
  "title_en": "title_en1",
  "content": r"""content1"""
},
1:{
  "env": "env2",
  "title": "title2",
  "title_en": "title_en2",
  "content": r"""content2"""
},
...
}
'''

prompt_template03 =r'''
你是一个工作细致的助手。我将给你一份数学教材（可能包含多个逻辑单元），请你帮我寻找出文本中所有的定义、定理、引理、推论、证明等逻辑单元，按照给定要求提取出对应内容，输出一个包含所有提取结果的 JSON 字典，每个逻辑单元对应一个子字典。。

具体提取的格式是：
{data_template}

你需要做的工作有以下内容：
1、准备一个json字典{{
  "env": "...",
  "title": "...",
  "title_en": "...",
  "content": r"""..."""
}}
2、在给定文本中进行识别，判定寻找出的内容是公理，定义，性质，定理，例子，引理，定理，推论，反例中的哪一种，并将判定的结果输入"env"的值。
3、将提取出的文本具体内容填入"content"的值，保留原文中的latex数学公式和环境，使用三引号对进行包裹，并加上r表示是原始字符串。
4、对所给文本内容进行概括和翻译得到标题（如："费马小定理的证明"，"关于函数自反性的一个例子"），标题中不要含有数学公式，将中文版填入"title"的值，将翻译后的英文版填入"title_en"的值。概括和翻译要简洁概括，体现核心内容。
5、搜索文本中所有公理，定义，性质，定理，例子，引理，定理，推论，反例，每一个这样的逻辑单元都占据一个json字典，他们的键是从0开始的编号，每次递增1.

提取过程中有以下要求：
1、可以包含数学公式，你输出的任何数学公式必须包含在完整正确的latex数学环境中，行间公式请使用"\["和"\]"，行内公式请使用"\("和"\)"    。
2、在工作期间，你将全程关闭搜索功能以及与外部的连接，仅凭文本本身内容来完成这项工作，不要擅自添加新的内容。
3、请你除了输出这个字典外，不要在你的输出开头和结尾添加其他的东西。
4、每个"content"的值必须使用三引号对进行包裹，并加上r表示是原始字符串。
5、表述题目的语言统一以中文输出，保持数学字符的格式不变。
6、"env"键的值必须且只能是公理，定义，性质，定理，例子，引理，定理，推论，反例其中一个。如果提取出的逻辑单元无法被其中任何一个概括，则不输出。
7、所有 JSON 键(env,title,title_en,content)必须保持不变。
8、"content"的值必须是一个定义/定理等完整的叙述，可以转化为符号数学语言。
  不能是'全迷向子空间'这样仅有一个名词;
  不能是'群是对称性的体现，它在代数结构的谱系中占据比环、域和向量空间更基本的地位。'、'回到一般的线性方程组. 形如 (1.3.1) 的写法现在显得有些累赘了, 不如引进较为紧凑的符号.'这样主观的表述，应使用精确的数学语言，可以被转化为符号数学语言；
  不能是'\\n这里用到了非零整数相乘依然非零这一性质,\\n"具有指代不明、前提和结论不完整的内容;
  不能有'满足(2.8.1)而非素数的正整数 $p$ 称为 carmichael 数'这样对编号的指代，你应将(2.8.1)内容代入，如果在文本中找不到，则不输出这样的内容。
9、如果给定文本中不含逻辑单元，请直接输出空集合。
10、请不要输出换行号"\n"，如果需要换行请使用latex中的换行符号。
11、请检查每个数学公式的latex环境是否完整正确，确保所有公式都能被正确编译，不要在公式的结尾缺少'\)'或'\]'。

示例：
Input:
定理 1.1.1 (代数基本定理) 设 $f = X^{{ n }} + a_{{ n - 1 }} X^{{ n - 1 }} + \cdots + a_{{ 0 }}$ 为以 $X$ 为变元的复系数 $n$ 次多项式, 其中 $n \in \mathbb{{ Z }}_{{ \geq 1 }}$ , 则存在 $x_{{ 1 }} , \ldots , x_{{ n }} \in \mathbb{{ C }}$ 使得

\[
f = \prod_{{ k = 1 }}^{{ n }} ( X - x_{{ k }} ) .
\]

这些 $x_{{ 1 }} , \ldots , x_{{ n }}$ 无非是多项式 $f$ 的复根 (计入重数); 精确到重排, 它们是唯一的.

特别地, 不仅限于二次多项式, 任意非常数多项式都有复数根.

Output:

{{0: {{
  "env": "定理",
  "title": "代数基本定理",
  "title_en": "Fundamental Theorem of Algebra",
  "content": r"""
设 $f = X^{{ n }} + a_{{ n - 1 }} X^{{ n - 1 }} + \cdots + a_{{ 0 }}$ 为以 $X$ 为变元的复系数 $n$ 次多项式, 其中 $n \in \mathbb{{ Z }}_{{ \geq 1 }}$ , 则存在 $x_{{ 1 }} , \ldots , x_{{ n }} \in \mathbb{{ C }}$ 使得

\[
f = \prod_{{ k = 1 }}^{{ n }} ( X - x_{{ k }} ) .
\]

这些 $x_{{ 1 }} , \ldots , x_{{ n }}$ 无非是多项式 $f$ 的复根 (计入重数); 精确到重排, 它们是唯一的.

特别地, 不仅限于二次多项式, 任意非常数多项式都有复数根.
"""
}}}}

以上是示例，请你不要在正文中输出。
############################################

以下是我给你的文本：{pos1}，请你帮我提取出文本中所有的定义、定理、引理、推论、命题，并放入一个字典。
'''

def validation03(text):
    return True

correction_prompt= '''
    你是一个严谨的校对员。我将给你一个由大模型生成的数据结构，请你根据规定格式内容进行校对和修正。
  
    校对的格式是：
    {data_template}
    同时请你校对"content"的内容,确保其中的latex环境和数学公式可以被正确的编译。
    以下是待校验的文本：{answer}，请你帮我校对和修正这段内容。
    '''

empty_template03='''
[]
'''


### 整理提取的逻辑单元

In [ ]:
def flatten_nested_dict(data: dict) -> dict:
    """
    将形如 {0:{}, 1:{0:{...}}, 2:{}, 8:{0:{...}}} 的嵌套字典
    转换为扁平化形式 {0:{...}, 1:{...}, 2:{...}} 并重新编号。
    """
    flat_dict = {}
    new_index = 0

    for outer_key, inner in data.items():
        if isinstance(inner, dict) and inner:  # 跳过空字典
            for _, value in inner.items():
                if isinstance(value, dict) and value:  # 确认是有效子字典
                    flat_dict[new_index] = value
                    new_index += 1

    return flat_dict

### 主函数-单个markdown文件的处理

In [ ]:
# In[15]:

def process_md(file_path):
    print(f'{file_path} is processing...')

    # 处理文本
    text_list = divider.divide(file_path)
    text_dict = {index: {"pos1": value} for index, value in enumerate(text_list)}

    # llm进行内容校对
    problem_corrector = MultiProcessor(llm=llm, parse_method=parser.parse_pads, data_template=data_template01, 
                                    prompt_template=prompt_template01, correction_template=correction_prompt, 
                                    validator=validation01, back_up_llm=None)
    corrected_text_dict = problem_corrector.multitask_perform(text_dict, num_threads=num_threads, checkpoint=checkpoint, 
                                                    Active_Reload=False, Active_Transform=False) 

    chopped_text_dict = pad_to_dict(corrected_text_dict)

    # 正则分句、分隔题目
    cut = MultiProcessor(llm=llm, parse_method=parser.parse_list, data_template=data_template02, 
                                    prompt_template=prompt_template02, correction_template=correction_prompt, 
                                    validator=validation02, back_up_llm=None)
    cut_marker = cut.multitask_perform(chopped_text_dict, num_threads=num_threads, checkpoint=checkpoint, 
                                                    Active_Reload=False, Active_Transform=False)

    problem_dict = extract_problem(chopped_text_dict, cut_marker)
    
    # 提取命题
    statement_extractor = MultiProcessor(llm=llm, parse_method=parser.parse_dict, data_template=data_template03, 
                                    prompt_template=prompt_template03, correction_template=correction_prompt, 
                                    validator=validation03, back_up_llm=None)
    statement_dict = statement_extractor.multitask_perform(problem_dict, num_threads=num_threads, checkpoint=checkpoint, 
                                                    Active_Reload=False, Active_Transform=False)
    
    sorted_statement_dict = flatten_nested_dict(statement_dict)

    # ✅ 保存为 JSON 文件（文件名 = 原 Markdown 文件名）
    base_filename = os.path.splitext(os.path.basename(file_path))[0]
    json_output_path = f"{base_filename}.json"
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(sorted_statement_dict, f, ensure_ascii=False, indent=4)

    return sorted_statement_dict

### 批量处理多个markdown文件

In [ ]:
# In[16]:


def multi_process_md(file_path):  
    results_list = []  
    
    for filename in os.listdir(file_path):  
        # 构建每个文件的完整路径  
        full_path = os.path.join(file_path, filename)  
        
        if os.path.isfile(full_path) and filename.endswith('.md'):  
            result_dict = process_md(full_path)  
            
            # 提取字典中的值部分，并更新到 results_list 中  
            results_list.extend(result_dict.values())  

    # 创建一个结果字典，使用 enumerate 为结果赋索引  
    results_dict = {index: value for index, value in enumerate(results_list)}  

    # 将结果字典写入 JSON 文件  
    with open('result_exam.json', 'w', encoding='utf-8') as json_file:  
        json.dump(results_dict, json_file, ensure_ascii=False, indent=4)  

    return results_dict  


In [ ]:
# In[17]:


def multi_process_md(file_path, output_folder):  
    results_list = []  
    
    # 检查输出文件夹是否存在，如果不存在则创建
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(file_path):  
        # 构建每个文件的完整路径  
        full_path = os.path.join(file_path, filename)  
        
        if os.path.isfile(full_path) and filename.endswith('.md'):  
            result_dict = process_md(full_path)  
            
            # 提取字典中的值部分，并更新到 results_list 中  
            results_list.extend(result_dict.values())  

            # 保存每个Markdown文件的解析结果为同名的JSON文件
            json_filename = os.path.splitext(filename)[0] + '.json'  # 获取JSON文件名
            json_path = os.path.join(output_folder, json_filename)  # 构建JSON文件的完整路径
            with open(json_path, 'w', encoding='utf-8') as json_file:  
                json.dump(result_dict, json_file, ensure_ascii=False, indent=4)  # 将解析结果写入JSON文件

    # 创建一个结果字典，使用 enumerate 为结果赋索引  
    results_dict = {index: value for index, value in enumerate(results_list)}  

    return results_dict  


### 转换为latex文件

In [ ]:

def json_to_latex(input_path, output_path):
    """
    将 JSON 字典文件转换为 LaTeX 源码文件。
    每个条目根据 env 转换为对应环境块。
    """

    # ====== 1. 读取 JSON 文件 ======
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # ====== 2. 定义环境映射 ======
    env_map = {
        "定义": "dfn",
        "定理": "thm",
        "命题": "ppt",
        "引理": "lma",
        "推论": "crl",
        "公理": "axm",
        "性质": "ppt",
        "例子": "xmp",
        "反例": "cxmp",
        "证明": "prf"
    }

    # ====== 3. 遍历条目并生成 LaTeX 块 ======
    output_blocks = []
    for idx, entry in data.items():
        env = entry.get("env", "").strip()
        env_short = env_map.get(env)

        # 如果没匹配上，跳过并提示
        if env_short is None:
            print(f"⚠️ 跳过条目 {idx}：未识别的环境类型 “{env}”")
            continue

        title = entry.get("title", "").strip()
        title_en = entry.get("title_en", "").strip()
        content = entry.get("content", "").strip().replace('\\n', '')
        label = title_en.replace("_", " ").replace("'", "").replace(",", "").replace(":", "").replace("(", "").replace(")", "").replace("$", "").replace("^", "").replace("{", "").replace("}", "").replace("\\", "")

        block = f"""
\\begin{{{env_short}}}
    [{label}]
    {{{title}}}
    [{title_en}]
    [gpt-4.1]
    {content}
\\end{{{env_short}}}

"""
        output_blocks.append(block)

    # ====== 4. 合并并写入文件 ======
    indented_blocks = ["    " + b.replace("\n", "\n    ") for b in output_blocks]
    result_text = " \n ".join(indented_blocks)

    header = r"""\documentclass[UTF8]{ctexart}

\usepackage{FulcrumCN}
\usepackage{geometry}
\usepackage{amssymb}
\geometry{
    paper =a4paper,
    top =3cm,
    bottom =3cm,
    left=2cm,
    right =2cm
}
\linespread{1.2}
\begin{document}
    \begin{center}
        {\LARGE\textbf{Fulcrum 自动生成示例}}

        SJTU AI4Math Team
    \end{center}

    \section{知识条目}
    \subsection{自动生成条目}
"""
    footer = r"""
    \section{附录}
    本文档由脚本自动生成，供 Fulcrum 数学知识条目测试使用。

\end{document}
"""
    result_text = header + "\n" + result_text + "\n" + footer

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(result_text)

    print(f"✅ 已成功生成 LaTeX 文件：{output_path}")


In [ ]:
#  处理所有 Markdown 文件并生成 LaTeX 文件
def process_all_md_to_latex(input_folder, output_folder):
    output_folder_json = os.path.join(output_folder, 'output_json')
    os.makedirs(output_folder_json, exist_ok=True)
    multi_process_md(input_folder, output_folder_json)
    for filename in os.listdir(output_folder_json):
        if filename.endswith('.json'):
            json_path = os.path.join(output_folder_json, filename)
            tex_path = os.path.join(output_folder, filename.replace('.json', '.tex'))
            json_to_latex(json_path, tex_path)

In [ ]:
test = process_all_md_to_latex('D:\\AI4Math\\JoinAgent\\test_input', 'D:\\AI4Math\\Fulcrum-Template')